## 1. 导入必要的库

In [4]:
# -*- coding: utf-8 -*-

import pandas as pd
import scipy.sparse as sps
import numpy as np
import os
import json
from tqdm import tqdm

from sklearn.model_selection import KFold

from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.Recommender_import_list import *
from RecSys_Course_AT_PoliMi.Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender
from Recommenders.BaseRecommender import BaseRecommender



## 2. 项目配置与常量

In [5]:
# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'model_output'
SUBMISSION_FOLDER = 'temp_output' # 提交文件的保存目录

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

print("项目配置加载完成.")

项目配置加载完成.


## 3. 辅助函数

### 3.1 加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.

In [6]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all


### 3.2 接收评估结果字典和模型名称，并以清晰的格式打印关键指标.

In [7]:
def print_results_formatted(results_df, model_name: str = "Model"):
    """
    接收评估结果 DataFrame 和模型名称，并以清晰的格式打印关键指标.

    Args:
        results_df (pd.DataFrame): 来自 Evaluator.evaluateRecommender() 的结果DataFrame.
        model_name (str): 要打印的模型名称.
    """
    # 检查 DataFrame 是否为空，或者指定的 cutoff 值是否作为索引存在
    if results_df.empty or EVALUATION_CUTOFF not in results_df.index:
        print(f"--- 在 cutoff={EVALUATION_CUTOFF} 处没有找到 '{model_name}' 的评估结果 ---")
        return

    # 使用 .loc[] 通过索引名来选取我们关心的那一行数据
    # 这会返回一个 Pandas Series，其行为非常像一个字典
    res_series = results_df.loc[EVALUATION_CUTOFF]

    print(f"--- 模型评估结果: {model_name} ---")
    print(f"--------------------------------------------------")
    # 现在在一个 Pandas Series 上使用 .get() 方法，这是安全且正确的
    print(f"{f'RECALL@{EVALUATION_CUTOFF}':<25}: {res_series.get('RECALL', -1):.4f}   <-- 竞赛官方指标")
    print(f"{f'PRECISION@{EVALUATION_CUTOFF}':<25}: {res_series.get('PRECISION', -1):.4f}")
    print(f"{f'MAP@{EVALUATION_CUTOFF}':<25}: {res_series.get('MAP', -1):.4f}")
    print(f"{f'HIT_RATE@{EVALUATION_CUTOFF}':<25}: {res_series.get('HIT_RATE', -1):.4f}")
    print(f"{f'ITEM_COVERAGE@{EVALUATION_CUTOFF}':<25}: {res_series.get('COVERAGE_ITEM', -1):.4f}")
    print(f"{f'AVG_POPULARITY@{EVALUATION_CUTOFF}':<25}: {res_series.get('AVERAGE_POPULARITY', -1):.4f}")
    print(f"--------------------------------------------------\n")

### 3.4 读取模型

In [8]:
def load_best_model(recommender_class, urm_train, file_name, modelfile_path):
    """
    加载一个由超参数搜索脚本保存的最佳模型。
    """
    file_path = os.path.join(modelfile_path, file_name)

    print(f"--- 正在加载预训练模型: {file_name} ---")

    # 检查模型文件是否存在
    if not os.path.exists(file_path + ".zip"):
        print(f">>> 警告: 模型文件 '{file_path}.zip' 不存在!")
        print(">>> 请确保超参数优化已完成，并且文件名正确。")
        return None

    # 1. 初始化一个“空”的模型对象
    recommender_instance = recommender_class(urm_train)

    # 2. 调用 .load_model() 方法加载数据
    recommender_instance.load_model(folder_path=modelfile_path, file_name=file_name)

    print("模型加载成功！\n")
    return recommender_instance

## 4. 分割数据集
本节将加载数据，并将其分割为训练集和验证集，为模型评估做准备。

In [9]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)

--- 正在分割数据用于本地验证... ---
数据分割完成.
训练集维度: (27095, 6969), 验证集维度: (27095, 6969)

EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
评估器初始化完成.


## 6. 模型融合

### 6a 双模型融合

#### 6a.1 定义融合推荐器

In [24]:
# --- 定义一个通用的混合推荐器类 ---
class HybridScoreRecommender(BaseRecommender):
    """
    一个通用的混合推荐器，通过加权平均多个模型的分数来进行推荐。
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridScoreRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 初始化一个零分矩阵
        final_scores = np.zeros((len(user_id_array), self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            # 获取单个模型的分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # 归一化分数
            scores = self._normalize_scores(scores)

            # 加权求和
            final_scores += self.weights[i] * scores

        return final_scores

    def _normalize_scores(self, scores):
        """对分数矩阵的每一行进行 Min-Max 归一化"""
        # 检查是否是稀疏矩阵，如果是，则转换为稠密数组
        if sps.issparse(scores):
            scores = scores.toarray()

        max_val = scores.max(axis=1, keepdims=True)
        min_val = scores.min(axis=1, keepdims=True)

        # 避免除以零
        denominator = max_val - min_val
        denominator[denominator == 0] = 1.0

        return (scores - min_val) / denominator

print("模型融合工具已准备就绪。")

模型融合工具已准备就绪。


#### 6a.2 加载已经训练好的两个最佳模型

In [ ]:
# --- 1. 加载模型 ---
# 定义模型所在的文件夹
COMBINE_MODEL_FOLDER = "temp_output"

print("--- 正在加载用于融合的基模型... ---")

# 加载 SLIMElasticNetRecommender
recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 加载 b_best_model
recommender_b = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train, # 使用 URM_train 初始化
    file_name="IALSRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# 确保两个模型都成功加载
if recommender_slim is None or recommender_b is None:
    print(">>> 错误：一个或多个基模型加载失败，无法继续进行融合。")
else:
    print("所有基模型均已成功加载！")

#### 6a.3 网格搜索寻找最佳融合权重

In [ ]:
# --- 2. 网格搜索寻找最佳融合权重 ---

best_recall = 0
best_alpha = 0
alpha_values = np.arange(0.65, 0.75, 0.01) # 以0.05为步长，进行更精细的搜索

# 确保模型已成功加载
if recommender_slim and recommender_b:
    print("\n--- 开始在本地验证集上寻找最佳融合权重 alpha ---")

    # 将模型实例放入一个列表中
    recommender_list = [recommender_slim, recommender_b]

    for alpha in tqdm(alpha_values, desc="搜索 Alpha 权重"):
        # alpha 是第一个模型的权重, (1-alpha) 是第二个模型的权重
        weights = [alpha, 1 - alpha]

        # 创建混合推荐器实例
        hybrid_recommender = HybridScoreRecommender(URM_train, recommender_list, weights)

        # 在验证集上评估
        results_df, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
        current_recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

        if current_recall > best_recall:
            best_recall = current_recall
            best_alpha = alpha
            print(f"发现新的最佳 Recall: {best_recall:.5f} (当 alpha = {best_alpha:.2f})")

    print(f"\n--- 搜索完成！---")
    print(f"最佳 Alpha (SLIMElasticNet 的权重): {best_alpha:.2f}")
    print(f"RP3beta 的权重: {(1 - best_alpha):.2f}")
    print(f"融合后在本地验证集上的最佳 Recall@20: {best_recall:.5f}")

#### 6a.4 使用最佳权重生成提交文件

In [ ]:
# --- 1. 配置最佳权重和模型信息 ---
BEST_ALPHA = 0.50 # !!! 请确保将此值替换为您在网格搜索中找到的最佳 alpha !!!

MODEL_FOLDER = 'model_output' # 模型文件所在的文件夹

# --- 2. 加载在 *全量数据* 上训练好的模型 ---
print("\n--- 正在加载在全量数据上预训练的最终模型... ---")

# 加载 SLIMElasticNetRecommender
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 加载 b
final_recommender_b = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="7-final_model_EASE_R_Recommender",
    modelfile_path=MODEL_FOLDER
)


# --- 3. 创建最终的混合推荐器并生成提交 ---
if final_recommender_slim and final_recommender_b:

    final_recommender_list = [final_recommender_slim, final_recommender_b]
    final_weights = [BEST_ALPHA, 1 - BEST_ALPHA]

    final_hybrid_recommender = HybridScoreRecommender(urm_all, final_recommender_list, final_weights)

    # --- 开始生成推荐 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终融合推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    submission_filename = f"submission_FINAL_Hybrid_SLIM_RP3_alpha_{BEST_ALPHA:.2f}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename) # SUBMISSION_FOLDER 在之前已定义

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
else:
    print("\n>>> 错误：一个或多个最终模型加载失败，无法生成提交文件。")

#### 6b.1 加载模型

### 6c 瀑布融合

#### 6c.1 加载所有模型

In [10]:
# MultiStageRecommender 类的定义保持不变
import numpy as np
import scipy.sparse as sps

class MultiStageRecommender(BaseRecommender):
    """
    一个实现了“召回与排序”架构的混合推荐器。

    - 召回模型 (candidate_generator): 一个快速、高召回率的模型，用于生成候选集。
    - 排序模型 (ranking_recommender): 一个精确、高准确率的模型，用于对候选集进行重新排序。
    """
    def __init__(self, urm_train, candidate_generator, ranking_recommender, candidate_cutoff=200):
        super(MultiStageRecommender, self).__init__(urm_train)
        self.candidate_generator = candidate_generator
        self.ranking_recommender = ranking_recommender
        self.candidate_cutoff = candidate_cutoff

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        final_scores = np.full((len(user_id_array), self.n_items), -np.inf, dtype=np.float32)

        for user_index, user_id in enumerate(user_id_array):
            candidate_items = self.candidate_generator.recommend(user_id,
                                                                 cutoff=self.candidate_cutoff,
                                                                 remove_seen_flag=True)
            if len(candidate_items) == 0:
                continue

            ranking_scores = self.ranking_recommender._compute_item_score(np.array([user_id]),
                                                                         items_to_compute=candidate_items)
            final_scores[user_index, candidate_items] = ranking_scores[0, candidate_items]
        return final_scores

print("MultiStageRecommender 类已定义。")

class EnsembleCandidateGenerator(BaseRecommender):
    """
    一个用于“多路召回”的推荐器。
    它会调用内部的多个召回模型，并将它们的推荐结果合并、去重，形成一个更大的候选集。
    """
    def __init__(self, urm_train, recommenders):
        super(EnsembleCandidateGenerator, self).__init__(urm_train)
        self.recommenders = recommenders

    def recommend(self, user_id, cutoff=100, remove_seen_flag=True, items_to_compute=None, remove_top_pop_flag=False, remove_custom_items_flag=False, return_scores=False):
        # 注意：这里的 cutoff 是指 *每个* 内部推荐器要召回的数量

        # 使用集合(set)来自动处理去重
        candidate_items_set = set()

        # 遍历所有的召回模型
        for recommender in self.recommenders:
            # 获取当前模型的推荐结果
            partial_recs = recommender.recommend(user_id,
                                                 cutoff=cutoff,
                                                 remove_seen_flag=remove_seen_flag)

            # 将结果添加到集合中
            candidate_items_set.update(partial_recs)

        # 将集合转换为列表返回
        # 注意：最终的候选集大小会小于 (模型数量 * cutoff)，具体取决于各模型结果的重合度
        return list(candidate_items_set)

    # 这个类不需要实现 _compute_item_score，因为它只用于生成候选列表
    def _compute_item_score(self, user_id_array, items_to_compute=None):
        raise NotImplementedError("EnsembleCandidateGenerator does not support score computation.")

class FusedRanker(BaseRecommender):
    """
    终极健壮的分数融合排序器：
    1. 健壮地处理分数归一化时的“全零”行。
    2. 使用 np.nan_to_num 清理上游模型可能返回的 NaN 或 inf 值。
    """
    def __init__(self, urm_train, recommenders, weights=None):
        super(FusedRanker, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights if weights is not None else [1/len(recommenders)] * len(recommenders)

    def _normalize_scores(self, scores):
        if sps.issparse(scores):
            scores = scores.toarray()

        for i in range(scores.shape[0]):
            row = scores[i]
            # 仅对有限数值进行计算，忽略 inf
            finite_vals = row[np.isfinite(row)]
            if len(finite_vals) == 0:
                # 如果行中所有值都是 inf 或 nan
                scores[i] = 0.0
                continue

            max_val = np.max(finite_vals)
            min_val = np.min(finite_vals)
            denominator = max_val - min_val

            if denominator == 0:
                scores[i] = 0.0
            else:
                scores[i] = (row - min_val) / denominator

        return scores

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        final_scores = np.zeros((len(user_id_array), self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            current_scores = recommender._compute_item_score(user_id_array, items_to_compute)

            normalized_scores = self._normalize_scores(current_scores)

            # 关键步骤：清理任何可能存在的 NaN 或 inf
            # 这可以防止它们污染最终的分数
            sanitized_scores = np.nan_to_num(normalized_scores)

            final_scores += self.weights[i] * sanitized_scores

        return final_scores

print("EnsembleCandidateGenerator 类已定义。")


# --- 1. 加载和准备所有用于网格搜索的候选模型 (在验证集上) ---
VALIDATION_MODEL_FOLDER = "temp_output"
print(f"\n--- 正在从 '{VALIDATION_MODEL_FOLDER}' 文件夹加载所有用于网格搜索的候选模型... ---")

# --- 准备排序模型候选人 (在 URM_train 上训练) ---
print("--- 准备排序模型 ---")
recommender_slim_val = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train,
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=VALIDATION_MODEL_FOLDER
)
recommender_ials_val = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=URM_train,
    file_name="IALSRecommender_best_model",
    modelfile_path=VALIDATION_MODEL_FOLDER
)

# --- 准备召回模型候选人 (在 URM_train 上训练) ---
# 注意：您的代码中没有提供 EASE_R 在 temp_output 的模型，我们假设它存在
# 如果不存在，您需要先运行EASE_R的超参数搜索
print("\n--- 准备召回模型 ---")
recommender_easer_val = load_best_model(
    recommender_class=EASE_R_Recommender, # 假设您有EASE_R的模型
    urm_train=URM_train,
    file_name="EASE_R_Recommender_best_model", # 假设这是它的文件名
    modelfile_path=VALIDATION_MODEL_FOLDER
)

recommender_itemknn_val = load_best_model(
    recommender_class=UserKNNCFRecommender, # 假设您有EASE_R的模型
    urm_train=URM_train,
    file_name="UserKNNCFRecommender_jaccard_best_model", # 假设这是它的文件名
    modelfile_path=VALIDATION_MODEL_FOLDER
)

MultiStageRecommender 类已定义。
EnsembleCandidateGenerator 类已定义。

--- 正在从 'temp_output' 文件夹加载所有用于网格搜索的候选模型... ---
--- 准备排序模型 ---
--- 正在加载预训练模型: SLIMElasticNetRecommender_best_model ---
SLIMElasticNetRecommender: Loading model from file 'temp_outputSLIMElasticNetRecommender_best_model'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: IALSRecommender_best_model ---
IALSRecommender: Loading model from file 'temp_outputIALSRecommender_best_model'
IALSRecommender: Loading complete
模型加载成功！


--- 准备召回模型 ---
--- 正在加载预训练模型: EASE_R_Recommender_best_model ---
EASE_R_Recommender: Loading model from file 'temp_outputEASE_R_Recommender_best_model'
EASE_R_Recommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: UserKNNCFRecommender_jaccard_best_model ---
UserKNNCFRecommender: Loading model from file 'temp_outputUserKNNCFRecommender_jaccard_best_model'
UserKNNCFRecommender: Loading complete
模型加载成功！



#### 6c.2 在本地验证集上寻找最佳召回

In [ ]:
# --- 2. 在本地验证集上进行网格搜索 (更新版) ---

models_ready = all([recommender_slim_val, recommender_ials_val, recommender_easer_val, recommender_itemknn_val])

if models_ready:
    # --- 1. 定义召回层候选人 (Candidate Generators) ---

    # 单模型召回
    cand_knn = recommender_itemknn_val
    cand_ials = recommender_ials_val
    cand_easer = recommender_easer_val

    # 多路召回 (并集)
    cand_ensemble_knn_ials = EnsembleCandidateGenerator(URM_train, [recommender_itemknn_val, recommender_ials_val])
    cand_ensemble_knn_easer = EnsembleCandidateGenerator(URM_train, [recommender_itemknn_val, recommender_easer_val])
    cand_ensemble_ials_easer = EnsembleCandidateGenerator(URM_train, [recommender_ials_val, recommender_easer_val])
    # 终极三合一召回
    cand_ensemble_all = EnsembleCandidateGenerator(URM_train, [recommender_itemknn_val, recommender_ials_val, recommender_easer_val])

    # 将所有召回器放入字典，方便遍历
    candidate_generators = {
        "C_ItemKNN": cand_knn,
        "C_IALS": cand_ials,
        "C_EASER": cand_easer,
        "C_Ens_KnnIals": cand_ensemble_knn_ials,
        "C_Ens_KnnEaser": cand_ensemble_knn_easer,
        "C_Ens_IalsEaser": cand_ensemble_ials_easer,
        "C_Ens_All": cand_ensemble_all,
    }
    print(f"已定义 {len(candidate_generators)} 种召回策略。")

    # --- 2. 定义排序层候选人 (Rankers) ---

    # 单模型排序
    rank_slim = recommender_slim_val
    rank_easer = recommender_easer_val
    rank_ials = recommender_ials_val

    # 分数融合排序 (这里我们使用平均融合，您也可以后续加入权重搜索)
    rank_fused_slim_easer = FusedRanker(URM_train, [recommender_slim_val, recommender_easer_val])
    rank_fused_slim_ials = FusedRanker(URM_train, [recommender_slim_val, recommender_ials_val])

    # 将所有排序器放入字典
    rankers = {
        "R_SLIM": rank_slim,
        "R_EASER": rank_easer,
        "R_IALS": rank_ials,
        "R_Fused_SlimEaser": rank_fused_slim_easer,
        "R_Fused_SlimIals": rank_fused_slim_ials,
    }
    print(f"已定义 {len(rankers)} 种排序策略。")

    # --- 3. 定义候选集大小 ---
    candidate_cutoffs = [100, 200, 300, 400]
    print(f"将测试 {len(candidate_cutoffs)} 种候选集大小。")

    # --- 开始全自动、大范围的网格搜索 ---

    best_recall = 0.0
    best_combination = {}
    results_log = []

    total_combinations = len(candidate_generators) * len(rankers) * len(candidate_cutoffs)
    print(f"\n--- 即将开始测试 {total_combinations} 种组合，请耐心等待... ---")

    pbar = tqdm(total=total_combinations, desc="网格搜索进度")

    for cand_name, cand_model in candidate_generators.items():
        for rank_name, rank_model in rankers.items():
            # 理论上，排序模型不应与召回模型中的任何一个相同，但我们先测试所有组合
            # if rank_name in cand_name:
            #     pbar.update(len(candidate_cutoffs))
            #     continue

            for k in candidate_cutoffs:

                # 构建“召回与排序”推荐器
                multistage_recommender = MultiStageRecommender(URM_train,
                                                               candidate_generator=cand_model,
                                                               ranking_recommender=rank_model,
                                                               candidate_cutoff=k)

                # 评估
                results_df, _ = evaluator_validation.evaluateRecommender(multistage_recommender)
                current_recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

                # 记录结果
                current_config = {
                    "Recall_Strategy": cand_name,
                    "Ranking_Strategy": rank_name,
                    "K": k,
                    "Recall@20": current_recall
                }
                results_log.append(current_config)

                if current_recall > best_recall:
                    best_recall = current_recall
                    best_combination = current_config
                    print(f"\n🔥🔥🔥 新的最佳组合! Recall: {best_recall:.5f} | Config: {cand_name} -> {rank_name} (K={k})")

                pbar.update(1)

    pbar.close()

    # --- 结果汇总 ---
    print("\n\n--- 网格搜索完成！---")
    print("找到的最佳组合如下:")
    print(json.dumps(best_combination, indent=4, ensure_ascii=False))

    # 将所有结果转换为DataFrame并按Recall排序，方便分析
    results_df_summary = pd.DataFrame(results_log)
    print("\n--- 所有组合性能排行 Top 10 ---")
    print(results_df_summary.sort_values(by="Recall@20", ascending=False).head(10))

    # 建议保存完整日志
    results_df_summary.to_csv("grid_search_log.csv", index=False)
    print("\n完整日志已保存至 grid_search_log.csv")

#### 6c.3 生成最终提交文件

In [11]:
# --- 1. 定义最终提交所需的参数和模型信息 ---

# 这是从网格搜索中得到的最佳组合
BEST_COMBINATION = {
    "召回策略": "C_Ens_KnnEaser",
    "排序策略": "R_Fused_SlimIals",
    "单模型召回数K": 400
}

# 模型文件所在的最终文件夹
FINAL_MODEL_FOLDER = 'model_output'

print("--- 最终提交配置 ---")
print(json.dumps(BEST_COMBINATION, indent=4, ensure_ascii=False))


# --- 2. 在 *全量数据* 上准备最佳组合所需的所有基础模型 ---

print(f"\n--- 正在从 '{FINAL_MODEL_FOLDER}' 文件夹或通过重新训练，准备最终模型... ---")

# --- 准备召回模型 (在 urm_all 上) ---
print("\n--- 准备召回层的模型 ---")
# a) EASE_R_Recommender (从文件加载)
print("加载 EASE_R Recommender...")
final_recommender_easer = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=urm_all,
    file_name="7-final_model_EASE_R_Recommender", # 确保这是您在全量数据上训练好的EASE_R模型文件名
    modelfile_path=FINAL_MODEL_FOLDER
)

# b) ItemKNNCF_jaccard (在全量数据上重新训练)
print("加载 ItemKNNCF (Jaccard)...")
final_recommender_itemknn = load_best_model(
    recommender_class=ItemKNNCFRecommender,
    urm_train=urm_all,
    file_name="11-final_model_ItemKNNCFRecommender", # 确保这是您在全量数据上训练好的ItemKNNCF模型文件名
    modelfile_path=FINAL_MODEL_FOLDER
)

# --- 准备排序模型 (在 urm_all 上) ---
print("\n--- 准备排序层的模型 ---")
# a) SLIMElasticNet (从文件加载)
print("加载 SLIMElasticNet Recommender...")
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all,
    file_name="5-1final_model_SLIMElasticNetRecommender-better", # 确保这是您在全量数据上训练好的SLIM模型文件名
    modelfile_path=FINAL_MODEL_FOLDER
)

# b) IALSRecommender (从文件加载)
print("加载 IALS Recommender...")
# 注意：您的初始代码中IALS模型在temp_output, 这里假设您也有一个在model_output中的最终版本
# 如果没有，请确保文件名和路径正确
final_recommender_ials = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all,
    file_name="5-2final_model_IALSRecommender", # 假设这是您在全量数据上训练好的IALS模型文件名
    modelfile_path=FINAL_MODEL_FOLDER
)


# --- 3. 构建最终的、分层的混合推荐系统 ---

# 检查所有模型是否都已成功准备就绪
all_models_ready = all([final_recommender_easer, final_recommender_itemknn, final_recommender_slim, final_recommender_ials])

if all_models_ready:
    print("\n--- 所有基础模型均已准备就绪，开始构建最终的推荐系统 ---")

    # a) 构建最终的多路召回器
    final_candidate_generator = EnsembleCandidateGenerator(urm_all,
                                                           recommenders=[final_recommender_itemknn, final_recommender_easer])
    print("最终的多路召回器 (ItemKNN + EASE_R) 已构建。")

    # b) 构建最终的分数融合排序器
    # 默认使用平均权重融合，因为我们在网格搜索中也是这么做的
    final_ranker = FusedRanker(urm_all,
                               recommenders=[final_recommender_slim, final_recommender_ials])
    print("最终的分数融合排序器 (SLIM + IALS) 已构建。")

    # c) 构建最终的“召回与排序”主推荐器
    final_recommender = MultiStageRecommender(urm_all,
                                              candidate_generator=final_candidate_generator,
                                              ranking_recommender=final_ranker,
                                              candidate_cutoff=BEST_COMBINATION["单模型召回数K"])
    print("最终的 '召回与排序' 推荐系统已构建完成！")


    # --- 4. 为目标用户生成并保存提交文件 ---

    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"\n--- 正在为 {len(target_user_ids)} 名目标用户生成最终推荐... ---")

    for user_id in tqdm(target_user_ids, desc="生成提交文件"):
        recommended_items = final_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF, # 最终推荐20个
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    submission_filename = (f"submission_FINAL_C_KnnEaser_R_SlimIals_K{BEST_COMBINATION['单模型召回数K']}.csv")
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename)

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n🎉🎉🎉 最终提交文件已成功生成! 🎉🎉🎉")
    print(f"文件保存在: {SUBMISSION_PATH}")

else:
    print("\n>>> 致命错误：一个或多个最终模型加载或训练失败，无法生成提交文件。")
    print(">>> 请检查 FINAL_MODEL_FOLDER 中的模型文件是否齐全且文件名正确。")

--- 最终提交配置 ---
{
    "召回策略": "C_Ens_KnnEaser",
    "排序策略": "R_Fused_SlimIals",
    "单模型召回数K": 400
}

--- 正在从 'model_output' 文件夹或通过重新训练，准备最终模型... ---

--- 准备召回层的模型 ---
加载 EASE_R Recommender...
--- 正在加载预训练模型: 7-final_model_EASE_R_Recommender ---
EASE_R_Recommender: Loading model from file 'model_output7-final_model_EASE_R_Recommender'
EASE_R_Recommender: Loading complete
模型加载成功！

加载 ItemKNNCF (Jaccard)...
--- 正在加载预训练模型: 11-final_model_ItemKNNCFRecommender ---
ItemKNNCFRecommender: Loading model from file 'model_output11-final_model_ItemKNNCFRecommender'
ItemKNNCFRecommender: Loading complete
模型加载成功！


--- 准备排序层的模型 ---
加载 SLIMElasticNet Recommender...
--- 正在加载预训练模型: 5-1final_model_SLIMElasticNetRecommender-better ---
SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

加载 IALS Recommender...
--- 正在加载预训练模型: 5-2final_model_IALSRecommender ---
IALSRecommender: Loading model from f

生成提交文件:   0%|          | 58/27095 [00:00<00:47, 568.36it/s]

最终的分数融合排序器 (SLIM + IALS) 已构建。
最终的 '召回与排序' 推荐系统已构建完成！

--- 正在为 27095 名目标用户生成最终推荐... ---


生成提交文件: 100%|██████████| 27095/27095 [01:00<00:00, 445.48it/s]


🎉🎉🎉 最终提交文件已成功生成! 🎉🎉🎉
文件保存在: temp_output\submission_FINAL_C_KnnEaser_R_SlimIals_K400.csv


### 6d 相似度融合

#### 6d.1 加载模型

In [ ]:
print("--- 正在加载用于相似度融合的基模型... ---")

# --- 加载模型 1: SLIMElasticNetRecommender ---
recommender_slim_val = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=URM_train,
    file_name="SLIMElasticNetRecommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# --- 加载模型 2: EASE_R_Recommender ---
recommender_easer_val = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=URM_train,
    file_name="EASE_R_Recommender_best_model",
    modelfile_path=COMBINE_MODEL_FOLDER
)

# --- 提取并统一相似度矩阵的格式 ---
if recommender_slim_val and recommender_easer_val:
    print("\n--- 正在提取并统一相似度矩阵格式... ---")

    # 1. 提取 SLIM 的稀疏矩阵 (它已经是稀疏的)
    W_slim = recommender_slim_val.W_sparse.copy()

    # 2. 提取 EASE_R 的稠密矩阵，并将其转换为 CSR 稀疏格式
    W_easer_dense = recommender_easer_val.W_sparse
    print(f"EASE_R 矩阵原始类型: {type(W_easer_dense)}")

    W_easer = sps.csr_matrix(W_easer_dense) # <--- 关键修正！
    print(f"EASE_R 矩阵已转换为: {type(W_easer)}")

    print("\n所有相似度矩阵已成功准备为稀疏格式！")
    print(f"  - SLIMElasticNet (非零元素: {W_slim.nnz})")
    print(f"  - EASE_R (非零元素: {W_easer.nnz})")
else:
    print("\n>>> 错误：一个或多个基模型准备失败，无法继续。")

#### 6d.2 相似度矩阵融合

In [ ]:
best_recall = 0
best_alpha = 0.0
# 以0.05为步长，进行精细的搜索
alpha_values = np.arange(0.90, 1.00, 0.05)
print("\n--- 开始在本地验证集上寻找最佳相似度融合权重 alpha ---")

for alpha in tqdm(alpha_values, desc="搜索 Alpha 权重"):
    # alpha 是 EASE_R 的权重, (1-alpha) 是 SLIM 的权重
    hybrid_similarity_matrix = (1 - alpha) * W_slim + alpha * W_easer

    # 创建并“训练”自定义相似度推荐器
    hybrid_recommender = ItemKNNCustomSimilarityRecommender(URM_train)
    hybrid_recommender.fit(hybrid_similarity_matrix)

    # 在验证集上评估
    results_df, _ = evaluator_validation.evaluateRecommender(hybrid_recommender)
    current_recall = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

    # 更新最佳结果
    if current_recall > best_recall:
        best_recall = current_recall
        best_alpha = alpha
        print(f"发现新的最佳 Recall: {best_recall:.5f} (当 alpha (EASE_R权重) = {best_alpha:.2f})")

print(f"\n--- 搜索完成！---")
print(f"最佳 Alpha (EASE_R 的权重): {best_alpha:.2f}")
print(f"SLIMElasticNet 的权重: {(1 - best_alpha):.2f}")
print(f"相似度融合后在本地验证集上的最佳 Recall@20: {best_recall:.5f}")

#### 6d.3 加载全量模型

In [55]:
# --- 1. 配置最佳权重和模型信息 ---

# !!! 关键步骤: 请将此值替换为您在本地验证中找到的最佳 alpha !!!
# alpha 是 EASE_R 的权重。根据您之前的实验，这个值很可能非常接近 1.0
BEST_ALPHA = 0.3  # 例如: BEST_ALPHA = 0.95

# --- 2. 加载在 *全量数据* 上训练好的模型 ---
print("\n--- 正在加载在全量数据上预训练的最终模型... ---")

# 加载 SLIMElasticNetRecommender (100% data version)
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 必须用 urm_all 初始化
    file_name="5-1final_model_SLIMElasticNetRecommender-better", # 请确保这是您保存的文件名
    modelfile_path=MODEL_FOLDER
)

# 加载 EASE_R_Recommender (100% data version)
final_recommender_easer = load_best_model(
    recommender_class=EASE_R_Recommender,
    urm_train=urm_all, # 必须用 urm_all 初始化
    file_name="7-final_model_EASE_R_Recommender", # 请确保这是您保存的文件名
    modelfile_path=MODEL_FOLDER
)

# --- 3. 提取、统一格式并融合相似度矩阵 ---
if final_recommender_slim and final_recommender_easer:

    print("\n--- 正在提取并融合最终的相似度矩阵... ---")

    # 提取 SLIM 的稀疏矩阵
    W_final_slim = final_recommender_slim.W_sparse

    # 提取 EASE_R 的稠密矩阵，并将其转换为 CSR 稀疏格式
    W_final_easer_dense = final_recommender_easer.W_sparse
    W_final_easer = sps.csr_matrix(W_final_easer_dense)

    # 使用最佳权重构建最终的混合相似度矩阵
    final_hybrid_similarity = (1 - BEST_ALPHA) * W_final_slim + BEST_ALPHA * W_final_easer

    print("最终混合相似度矩阵构建完成。")

    # --- 4. 创建最终的推荐器 ---
    final_hybrid_recommender = ItemKNNCustomSimilarityRecommender(urm_all)
    final_hybrid_recommender.fit(final_hybrid_similarity)


    # --- 5. 生成提交文件 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终融合推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 6. 保存文件 ---
    submission_filename = f"submission_SimFusion_SLIM_EASER_alpha_{BEST_ALPHA:.2f}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename)

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")

else:
    print("\n>>> 错误：一个或多个最终模型加载失败，无法生成提交文件。")


--- 正在加载在全量数据上预训练的最终模型... ---
--- 正在加载预训练模型: 5-1final_model_SLIMElasticNetRecommender-better ---
SLIMElasticNetRecommender: Loading model from file 'model_output5-1final_model_SLIMElasticNetRecommender-better'
SLIMElasticNetRecommender: Loading complete
模型加载成功！

--- 正在加载预训练模型: 7-final_model_EASE_R_Recommender ---
EASE_R_Recommender: Loading model from file 'model_output7-final_model_EASE_R_Recommender'
EASE_R_Recommender: Loading complete
模型加载成功！


--- 正在提取并融合最终的相似度矩阵... ---



  0%|          | 0/27095 [00:00<?, ?it/s]

最终混合相似度矩阵构建完成。
--- 正在为 27095 名目标用户生成最终融合推荐... ---



100%|██████████| 27095/27095 [00:42<00:00, 643.00it/s]



--- 最终提交文件已成功生成! ---
文件保存在: temp_output\submission_SimFusion_SLIM_EASER_alpha_0.99.csv


## 7. 生成最终提交文件
当你通过本地验证找到了最佳模型和参数后，在这里使用 **全部训练数据** (`urm_all`) 进行训练，并为测试集用户生成推荐。

### 7a. 重新训练以生成提交文件

In [ ]:
# --- 模型配置 ---
# 取消注释你想要使用的模型，并确保参数是最佳的

# # 配置
model_config = {
    "class": EASE_R_Recommender,
    "params": {"topK": None, "normalize_matrix": False, "l2_norm": 317.4516467206718}
}


# 自动生成与模型相关的文件名，便于维护
model_name = model_config['class'].RECOMMENDER_NAME
submission_filename = f"submission_{model_name}.csv"
model_filename = f"final_model_{model_name}"
params_filename = f"final_params_{model_name}.json"

SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename)
MODEL_SAVE_PATH = os.path.join(MODEL_FOLDER, model_filename)
PARAMS_SAVE_PATH = os.path.join(MODEL_FOLDER, params_filename)


# -----------------------------------------------------------------------------
# STEP 2: 执行生成流程 (通常无需修改以下代码)
# -----------------------------------------------------------------------------

# 确保输出和模型文件夹存在
for folder in [MODEL_FOLDER, SUBMISSION_FOLDER]:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"文件夹 '{folder}' 已创建。")

# 1. 确定最终模型类和参数
final_model_class = model_config["class"]
final_model_params = model_config["params"]

# 2. 在 *全部* 数据上训练最终模型
print(f"--- 正在使用模型 '{model_name}' 在全量数据上进行训练... ---")
# 警告：对于 SLIMElasticNet 等模型，这一步会非常耗时!
final_recommender = final_model_class(urm_all)
final_recommender.fit(**final_model_params)
print("最终模型训练完成。\n")


# 3. ***** 新增功能: 保存模型和参数 *****
print(f"--- 正在保存已训练的模型和参数... ---")
# 保存模型对象
final_recommender.save_model(folder_path=MODEL_FOLDER, file_name=model_filename)
# 保存参数字典为 JSON 文件
with open(PARAMS_SAVE_PATH, 'w') as f:
    json.dump(final_model_params, f, indent=4)
print(f"模型已保存至: {MODEL_SAVE_PATH}.zip")
print(f"参数已保存至: {PARAMS_SAVE_PATH}\n")


# 4. 加载需要预测的目标用户
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

# 5. 生成推荐并保存到文件
print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
submission = []
for user_id in tqdm(target_user_ids):
    recommended_items = final_recommender.recommend(
        user_id,
        cutoff=EVALUATION_CUTOFF,
        remove_seen_flag=True
    )
    submission.append((user_id, ' '.join(map(str, recommended_items))))

# 6. 创建DataFrame并保存为CSV
df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
df_submission.to_csv(SUBMISSION_PATH, index=False)

print(f"\n--- 提交文件已成功生成! ---")
print(f"文件保存在: {SUBMISSION_PATH}")
print("\n文件预览:")
print(df_submission.head())

### 7b. 读取模型以获得提交文件

In [ ]:
# --- STEP 1: 加载预训练的最佳模型 ---

# 使用我们之前编写的 load_best_model 辅助函数
# 注意：这里传入的是 urm_all，因为 recommend 函数需要访问完整的交互历史来移除已看过的物品
# 而模型本身是在 URM_train 上训练的，这些信息都保存在了模型文件中。
# 实际上，初始化时传入哪个 URM 并不影响加载，但传入 urm_all 在逻辑上更清晰。
final_recommender = load_best_model(SLIMElasticNetRecommender, urm_all)


if final_recommender:
    # --- STEP 2: 执行生成流程 ---

    # 1. 加载需要预测的目标用户
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    # 2. 生成推荐并保存到文件
    submission = []
    # 为了让输出更美观，我们可以使用 tqdm 来显示进度条
    from tqdm import tqdm

    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # 3. 创建DataFrame并保存为CSV
    submission_filename = f"submission_{final_recommender.RECOMMENDER_NAME}_optimized.csv"
    SUBMISSION_PATH = os.path.join(OUTPUT_FOLDER, submission_filename)

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
    print("\n文件预览:")
    print(df_submission.head())

else:
    print(">>> 模型加载失败，无法生成提交文件。请检查文件路径和名称。")